In [3]:
import importlib       # 👈 primero importa la libreria
import ui_main         # importa tu modulo
importlib.reload(ui_main)   # fuerza la recarga del archivo
from ui_main import Simultext   # 👈 mismo nombre de la clase
import tkinter as tk


def main():
    root = tk.Tk()
    root.title("SIMULTEXT")
    app = Simultext(root)
    root.mainloop()

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'xmltodict'

In [ ]:
class VentanaEtiquetadoParrafos:
    def __init__(self, root, ruta_xml, tipo_texto):
        # ... código existente ...
        
        # Agregar para tipos de discurso
        self.tipos_discurso_seleccionados = []  # Lista para múltiples selecciones
        self.tipos_discurso_disponibles = list(config.TIPOS_DISCURSO.keys())
        
        # ... resto del código ...

    def crear_interfaz(self):
        # ... código existente ...
        
        # === NUEVA SECCIÓN - TIPO DE DISCURSO ===
        marco_discurso = ttk.Frame(right_frame, style='Fondo.TFrame')
        marco_discurso.pack(fill=tk.X, pady=15, padx=5)
        
        ttk.Label(
            marco_discurso, 
            text="TIPO DE DISCURSO", 
            font=("Arial", 12, "bold"),
            foreground=config.COLOR_TITULO,
            background=config.COLOR_FONDO
        ).pack(anchor="w", pady=5)
        
        # Frame para checkboxes
        frame_checks = ttk.Frame(marco_discurso, style='Fondo.TFrame')
        frame_checks.pack(fill=tk.X)
        
        # Crear checkboxes en 2 columnas
        self.vars_discurso = {}  # Diccionario para guardar las variables
        fila, columna = 0, 0
        for i, tipo in enumerate(self.tipos_discurso_disponibles):
            var = tk.BooleanVar(value=False)
            self.vars_discurso[tipo] = var
            
            check = ttk.Checkbutton(
                frame_checks,
                text=tipo,
                variable=var,
                style='Fondo.TCheckbutton'
            )
            check.grid(row=fila, column=columna, sticky="w", padx=5, pady=2)
            
            # Alternar columnas
            columna += 1
            if columna > 1:
                columna = 0
                fila += 1
        
        # ... resto del código ...

    def configurar_estilos(self):
        # ... código existente ...
        style.configure('Fondo.TCheckbutton', background=config.COLOR_FONDO)

    def guardar_etiquetas(self):
        try:
            # Obtener tipos de discurso seleccionados
            tipos_discurso_seleccionados = []
            for tipo, var in self.vars_discurso.items():
                if var.get():
                    codigo = config.TIPOS_DISCURSO[tipo]
                    tipos_discurso_seleccionados.append(codigo)
            
            # ... resto del código de guardado ...
            
            datos_completos = {
                "metadata": {
                    "tipo_texto": self.tipo_texto,
                    "archivo_origen": self.ruta_xml,
                    "tipos_discurso": tipos_discurso_seleccionados  # ← NUEVO
                },
                "etiquetas": etiquetas_guardar
            }
            
            # ... resto del código ...

In [ ]:
config.er

['introduccion',
 'desarrollo',
 'conclusion',
 'datos_bibliograficos',
 'titulo',
 'subtitulo',
 'datos_autor',
 'referencia']

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

# Ruta de tu XML
ruta_xml = "/home/ospina/General/SIMULTEX/PrototipoEtiquetadoReferentes/simultext_modular/data/prueba.xml"

# Parsear XML
tree = ET.parse(ruta_xml)
root = tree.getroot()

# Contenedores
texto_reconstruido = []
filas = []

# Recorrer parrafos
for parrafo in root.findall(".//paragraph"):
    id_parrafo = parrafo.get("id")

    for oracion in parrafo.findall(".//sentence"):
        id_oracion = oracion.get("id")
        tokens = []

        for token in oracion.findall(".//token"):
            id_token = token.get("id")
            forma = token.get("form", "")
            lema = token.get("lemma", "")
            pos = token.get("pos", "")

            # Guardar para reconstrucción del texto
            tokens.append(forma)

            # Guardar para tabla
            filas.append({
                "id_parrafo": id_parrafo,
                "id_oracion": id_oracion,
                "id_token": id_token,
                "forma": forma,
                "lema": lema,
                "pos": pos
            })

        # Reconstruir oracion con trazabilidad
        oracion_texto = f"[{id_parrafo} - {id_oracion}] " + " ".join(tokens)
        texto_reconstruido.append(oracion_texto)

# Mostrar texto reconstruido
print("=== Texto reconstruido ===\n")
print("\n".join(texto_reconstruido[:10]))  # muestras las primeras 10 oraciones



=== Texto reconstruido ===

[p1 - s1] Chile necesita más ciencia y más científicos
[p2 - s2] Por Max_Bañados , decano Facultad_de_Física_de_la_Pontifica_Universidad_Católica_de_Chile
[p3 - s3] La ciencia - o el interés por conocer y entender el funcionamiento de el mundo que nos rodea - es una de las actividades más antiguas y primarias de el ser humano .
[p3 - s4] Las civilizaciones más antiguas ya buscaban estudiar y describir los procesos que modulan lo vivo y su entorno .
[p3 - s5] Hoy día la ciencia está presente en todas las actividades de la sociedad moderna .
[p3 - s6] Los rápidos y muy significativos avances recientes no significan que el interés por conocer se haya acabado , o que las preguntas se hayan agotado .
[p3 - s7] La espiral de el conocimiento no acaba nunca .
[p3 - s8] Por_el_contrario , los hechos muestran que detrás_de cada descubrimiento hay una nueva gran pregunta .
[p3 - s9] La teoría de la Evolución o la Relatividad dieron respuesta a muchas interrogantes , pe

In [2]:
import xml.etree.ElementTree as ET
import pandas as pd
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import json

class EtiquetadorApp:
    def __init__(self, root, ruta_xml):
        self.root = root
        self.root.title("Sistema de Etiquetado XML")
        self.root.geometry("1400x800")
        
        # Configurar el color de fondo general
        self.bg_color = '#f0f0f0'  # Color gris claro típico de tkinter
        self.root.configure(bg=self.bg_color)
        
        self.ruta_xml = ruta_xml
        self.parrafos = []
        self.etiquetas_parrafos = {}
        
        self.etiquetas_disponibles = [
            "titulo", 
            "subtitulo", 
            "datos bibliograficos", 
            "datos del autor", 
            "introduccion", 
            "desarrollo", 
            "conclusion", 
            "referencia"
        ]
        
        self.cargar_datos_xml()
        self.crear_interfaz()

    def cargar_datos_xml(self):
        try:
            tree = ET.parse(self.ruta_xml)
            root = tree.getroot()
            self.parrafos = []

            for parrafo in root.findall(".//paragraph"):
                id_parrafo = parrafo.get("id")
                texto_parrafo = ""

                for oracion in parrafo.findall(".//sentence"):
                    tokens = []
                    for token in oracion.findall(".//token"):
                        forma = token.get("form", "")
                        tokens.append(forma)
                    oracion_texto = " ".join(tokens)
                    texto_parrafo += oracion_texto + " "

                self.parrafos.append({
                    "id": id_parrafo,
                    "texto": texto_parrafo.strip()
                })
                
                self.etiquetas_parrafos[id_parrafo] = ""
                    
        except Exception as e:
            messagebox.showerror("Error", f"No se pudo cargar el XML: {str(e)}")

    def crear_interfaz(self):
        # Frame principal
        main_frame = ttk.PanedWindow(self.root, orient=tk.HORIZONTAL)
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        # Frame izquierdo - Texto completo
        left_frame = ttk.Frame(main_frame)
        main_frame.add(left_frame, weight=2)

        # Frame derecho - Etiquetado
        right_frame = ttk.Frame(main_frame)
        main_frame.add(right_frame, weight=1)

        # === LADO IZQUIERDO - TEXTO COMPLETO ===
        ttk.Label(left_frame, text="Texto Completo", font=("Arial", 14, "bold")).pack(pady=10)
        
        # Área de texto con fondo gris
        self.texto_area = scrolledtext.ScrolledText(
            left_frame, 
            wrap=tk.WORD, 
            font=("Arial", 11),
            bg=self.bg_color,  # Fondo gris
            relief=tk.FLAT,    # Sin borde resaltado
            borderwidth=1
        )
        self.texto_area.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        self.mostrar_texto_completo()

        # === LADO DERECHO - ETIQUETADO DE PÁRRAFOS ===
        ttk.Label(right_frame, text="Etiquetado de Párrafos", font=("Arial", 14, "bold")).pack(pady=10)
        
        parrafos_frame = ttk.Frame(right_frame)
        parrafos_frame.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)

        # Canvas y scrollbar
        canvas = tk.Canvas(parrafos_frame, bg=self.bg_color)
        scrollbar = ttk.Scrollbar(parrafos_frame, orient="vertical", command=canvas.yview)
        scrollable_frame = ttk.Frame(canvas)

        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )

        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)

        # Widgets para cada párrafo
        self.widgets_parrafos = {}
        for parrafo in self.parrafos:
            frame_parrafo = ttk.Frame(scrollable_frame, relief="solid", borderwidth=1)
            frame_parrafo.pack(fill=tk.X, pady=8, padx=5)
            
            ttk.Label(frame_parrafo, text=f"Párrafo {parrafo['id']}", 
                     font=("Arial", 10, "bold")).pack(anchor="w", padx=5, pady=2)
            
            texto_corto = parrafo['texto'][:150] + "..." if len(parrafo['texto']) > 150 else parrafo['texto']
            label_texto = ttk.Label(frame_parrafo, text=texto_corto, wraplength=350, justify="left")
            label_texto.pack(anchor="w", padx=5, pady=3)
            
            combo_frame = ttk.Frame(frame_parrafo)
            combo_frame.pack(fill=tk.X, padx=5, pady=5)
            
            ttk.Label(combo_frame, text="Etiqueta:").pack(side=tk.LEFT)
            
            combo_var = tk.StringVar(value=self.etiquetas_parrafos[parrafo['id']])
            combo = ttk.Combobox(combo_frame, textvariable=combo_var, 
                                values=self.etiquetas_disponibles, state="readonly", width=20)
            combo.pack(side=tk.LEFT, padx=5)
            
            self.widgets_parrafos[parrafo['id']] = combo_var

        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")

        # Botones
        botones_frame = ttk.Frame(right_frame)
        botones_frame.pack(pady=15, padx=5, fill=tk.X)
        
        self.btn_guardar = ttk.Button(botones_frame, text="Guardar", command=self.guardar_etiquetas)
        self.btn_guardar.pack(side=tk.LEFT, padx=5, expand=True)
        
        self.btn_cancelar = ttk.Button(botones_frame, text="Cancelar", command=self.root.quit)
        self.btn_cancelar.pack(side=tk.LEFT, padx=5, expand=True)

    def mostrar_texto_completo(self):
        texto_formateado = ""
        
        for parrafo in self.parrafos:
            #texto_formateado += f"\n{'='*80}\n"
            #texto_formateado += f"PÁRRAFO {parrafo['id']}\n"
            #texto_formateado += f"{'='*80}\n\n"
            texto_formateado += f"PÁRRAFO {parrafo['id'].upper()}\n - {parrafo['texto']}\n\n"
        
        self.texto_area.delete(1.0, tk.END)
        self.texto_area.insert(tk.END, texto_formateado.strip())
        self.texto_area.config(state=tk.DISABLED)

    def guardar_etiquetas(self):
        try:
            etiquetas_guardar = {}
            parrafos_sin_etiqueta = []
            
            for parrafo_id, combo_var in self.widgets_parrafos.items():
                etiqueta = combo_var.get()
                if etiqueta:
                    etiquetas_guardar[parrafo_id] = etiqueta
                    self.etiquetas_parrafos[parrafo_id] = etiqueta
                else:
                    parrafos_sin_etiqueta.append(parrafo_id)
            
            if not etiquetas_guardar:
                messagebox.showwarning("Etiquetado incompleto", 
                                      "No hay párrafos etiquetados. Por favor, asigna etiquetas a al menos un párrafo antes de guardar.")
                return
            
            if parrafos_sin_etiqueta:
                respuesta = messagebox.askyesno(
                    "Etiquetado incompleto", 
                    f"Hay {len(parrafos_sin_etiqueta)} párrafos sin etiquetar.\n\n¿Deseas guardar solo los párrafos etiquetados?"
                )
                if not respuesta:
                    return
            
            with open("etiquetas_parrafos.json", "w", encoding="utf-8") as f:
                json.dump(etiquetas_guardar, f, ensure_ascii=False, indent=2)
            
            resumen = f"Etiquetas guardadas exitosamente!\n\n"
            resumen += f"Párrafos etiquetados: {len(etiquetas_guardar)}\n"
            resumen += f"Párrafos sin etiquetar: {len(parrafos_sin_etiqueta)}\n\n"
            resumen += "Contenido del archivo:\n"
            resumen += json.dumps(etiquetas_guardar, indent=2, ensure_ascii=False)
            
            messagebox.showinfo("Guardado exitoso", resumen)
            
        except Exception as e:
            messagebox.showerror("Error", f"No se pudieron guardar las etiquetas: {str(e)}")

# Función principal
def main():
    ruta_xml = "/home/ospina/General/SIMULTEX/PrototipoEtiquetadoReferentes/simultext_modular/data/prueba.xml"
    
    root = tk.Tk()
    app = EtiquetadorApp(root, ruta_xml)
    root.mainloop()

if __name__ == "__main__":
    main()